# 🎬 Telugu Dialogue SLM Zero-Shot Benchmark: Gemma-2 vs. Sarvam-2B
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaiKiranKumarNalla/tenglish-lm/blob/main/notebooks/02_Telugu_Dialogue_SLM_Benchmark.ipynb)

**Objective:** Test whether off-the-shelf foundation SLMs (**Gemma-2-2B / 9B** and Indic foundation **Sarvam-2B**) can accurately predict Telugu movie dialogue continuations, capture Trivikram witty antithesis, Boyapati faction dialect, and handle English loanwords/Tenglish.

In [ ]:
# @title Step 1: Install Dependencies (Transformers, BitsAndBytes 4-bit, Accelerate)
!pip install -q transformers accelerate bitsandbytes sentencepiece protobuf torch rouge-score

In [ ]:
# @title Step 2: Load Ground-Truth Benchmark Dataset from Google Drive
import json, glob, os, csv
from google.colab import drive

# Mount Google Drive if available
try:
    drive.mount('/content/drive')
    CORPUS_DIR = '/content/drive/MyDrive/Tenglish_LM/movie_corpus'
except Exception:
    CORPUS_DIR = './corpus_data'

print(f"Corpus location: {CORPUS_DIR}")

In [ ]:
# @title Step 3: Select Models to Test
# Options: 'google/gemma-2-2b-it' (Lightweight Google SLM) or 'sarvamai/sarvam-2b' (India-specific 2B SLM)
MODEL_ID = "google/gemma-2-2b-it" # @param ["google/gemma-2-2b-it", "sarvamai/sarvam-2b", "Qwen/Qwen2.5-1.5B-Instruct"]
LOAD_IN_4BIT = True # @param {type:"boolean"}

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = None
if LOAD_IN_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

print(f"⏳ Loading model '{MODEL_ID}' on GPU...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
print(f"✅ Model '{MODEL_ID}' loaded successfully!")

In [ ]:
# @title Step 4: Fair & Unbiased Benchmark Evaluation (Context-Conditioned & Perplexity Scoring)
# Eliminating blind guessing bias:
# 1. Provide exact situational scene background & character motivation.
# 2. Calculate Perplexity (Cross-Entropy Loss) on the Ground Truth text given the context.

import math

BENCHMARK_SCENES = [
    {
        "movie": "Aravinda_Sametha",
        "director": "Trivikram Srinivas",
        "scene_synopsis": "ఫ్యాక్షన్ హింస తర్వాత కోడలి బొట్టు తుడిచేసిన తీవ్రమైన శోక సందర్భం. నాయన చనిపోయిన బాధలో కొడుకు కత్తి పట్టుకుంటాడేమోనని ఆందోళన.",
        "emotion": "ANGER_WARNING (Rayalaseema Faction Dialect)",
        "dialect": "RAYALASEEMA",
        "context": [
            "మీ నాయన అట్టపోయినాడు నువ్వు ఇట్టు పోతుండావా ఈనే ఉంటే ఆయన పోయాడనే బాధో ఆయన్ని చంపినారనే కోపం తప్ప",
            "వేరే దారిక కనపడట్లేదమ్మా ఎతకని అయినోళ్ళ కోసం కత్తి తూసాను. అదే అయనోళ్ళ కోసం కత్తి వదలలేనా"
        ],
        "ground_truth": "వెళ్లి తీరా"
    },
    {
        "movie": "Atadhu",
        "director": "Trivikram Srinivas",
        "scene_synopsis": "ఒక ప్రొఫెషనల్ కిల్లర్ మరియు రాజకీయ నాయకుడి కుట్ర. ప్రజల సానుభూతిని పొందడానికి తనపై తానే నకిలీ హత్యాయత్నం చేయించుకోవాలనే ఆలోచన.",
        "emotion": "CONVERSATIONAL_DRAMA / INTELLECTUAL CRIME",
        "dialect": "STANDARD_TELUGU",
        "context": [
            "ఒక అబ్బాయి పదో తరగతి ఫెయిల్ అయ్యి ఇంటికి వస్తే ఇంట్లో అందరూ అతన్ని తిట్టారు. ఆత్మహత్యాయత్నం చేస్తే ఫలితం చూసి ప్రజలు అతని ప్రాణాలను కాపాడి ఇంటికి తీసుకువస్తే ఇంట్లో అందరూ అతన్ని ఓదార్చారు.",
            "మొదటి సందర్భంలో విఫలమైనందుకు ప్రజలు అతనిపై కోపంగా ఉంటారు. రెండవ సందర్భంలో ఆత్మహత్యాయత్నం నుండి బయటపడినందుకు సానుభూతి ఉంటుంది."
        ],
        "ground_truth": "నాకు అది కావాలి. సానుభూతి! కాబట్టి నా జీవితంపై ఒక ప్రయత్నం జరగాలి. కానీ నేను దానిని తట్టుకుని నిలబడాలి."
    },
    {
        "movie": "Gabbar_Singh",
        "director": "Harish Shankar",
        "scene_synopsis": "పోలీస్ స్టేషన్ లో మాస్ పోలీస్ ఆఫీసర్ గబ్బర్ సింగ్ గూండాలను బెదిరిస్తూ తన అధికారిక ఆధిపత్యాన్ని చాటుకునే సన్నివేశం.",
        "emotion": "COMEDY / MASS_ELEVATION",
        "dialect": "STANDARD_TELUGU",
        "context": [
            "నాకు తిక్క ఉంది.. కానీ దానికి ఒక లెక్క ఉంది!",
            "పోలీస్ స్టేషన్ లో ఎవరు హీరో?"
        ],
        "ground_truth": "గబ్బర్ సింగ్!"
    }
]

for idx, item in enumerate(BENCHMARK_SCENES, start=1):
    prompt = (
        f"[సినిమా: {item['movie']}]\n"
        f"[దర్శకత్వం: {item['director']}]\n"
        f"[సన్నివేశం నేపథ్యం: {item['scene_synopsis']}]\n"
        f"[భావోద్వేగం: {item['emotion']} | మాండలికం: {item['dialect']}]\n\n"
        f"ఈ సన్నివేశం తాలూకు సహజమైన తెలుగు సంభాషణను కొనసాగించండి:\n"
    )
    for turn_no, turn in enumerate(item["context"], start=1):
        prompt += f"పాత్ర {turn_no}: {turn}\n"
    prompt += f"పాత్ర {len(item['context']) + 1}:"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=60, temperature=0.7, top_p=0.9, do_sample=True)
    pred = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # Unbiased Metric: Calculate Perplexity (Cross-Entropy Loss) on the Ground Truth text given the context
    full_text = prompt + " " + item["ground_truth"]
    full_enc = tokenizer(full_text, return_tensors="pt").to("cuda")
    target_ids = full_enc.input_ids.clone()
    target_ids[:, :inputs.input_ids.shape[1]] = -100 # Mask out prompt tokens

    with torch.no_grad():
        loss = model(full_enc.input_ids, labels=target_ids).loss
        ppl = math.exp(loss.item()) if loss.item() < 20 else float("inf")

    print(f"\n{'='*75}\n[TEST {idx}] {item['movie']} ({item['director']})\n{'='*75}")
    print(f"SCENE CONTEXT & BACKGROUND:\n  {item['scene_synopsis']}\n")
    print("PROMPT CONTEXT:")
    for turn in item["context"]:
        print(f"  - {turn}")
    print(f"\nGROUND TRUTH TARGET:\n  🎯 \"{item['ground_truth']}\"")
    print(f"\nMODEL GENERATION ({MODEL_ID}):\n  👉 \"{pred}\"")
    print(f"\nUNBIASED LIKELIHOOD METRIC:\n  📊 Ground-Truth Cross-Entropy Loss: {loss.item():.4f} | Perplexity (PPL): {ppl:.2f}")
    print(f"  (Lower PPL means the model expected this exact Telugu dialogue!)")
